# Lab Manager Web App

Este notebook solo sirve para cargar el paquete y abrir la aplicacion web en un puerto.

Archivos necesarios en la misma carpeta de Drive:

- `Lab_App_Colab_UI.ipynb`
- `lab_pipeline_package.zip`

Ejecuta las dos celdas. La segunda abrira una ventana/pestana con la app visual.


## 1. Setup
Monta Drive, instala dependencias y carga `lab_pipeline`.


In [ ]:
from pathlib import Path
import os
import sys
import subprocess
import zipfile

PACKAGE_ZIP = 'lab_pipeline_package.zip'
PACKAGE_ZIP_PREFIX = 'lab_pipeline_package'
NOTEBOOK_NAMES = ['Lab_App_Colab_UI.ipynb', 'Lab_Main_Pipeline.ipynb']

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'openpyxl', 'reportlab', 'pandas', 'flask'])

def running_in_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except Exception:
        return False

def mount_drive_if_colab():
    if running_in_colab():
        from google.colab import drive
        drive.mount('/content/drive')

def find_notebook_workspace():
    if not running_in_colab():
        return Path.cwd()
    mydrive = Path('/content/drive/MyDrive')
    matches = []
    for notebook_name in NOTEBOOK_NAMES:
        matches.extend(mydrive.rglob(notebook_name))
    if matches:
        return max(matches, key=lambda path: path.stat().st_mtime).parent
    workspace = mydrive / 'Generador_GNT'
    workspace.mkdir(parents=True, exist_ok=True)
    return workspace

def force_workspace_first(workspace_dir):
    workspace_text = str(workspace_dir)
    sys.path[:] = [path for path in sys.path if path != workspace_text]
    sys.path.insert(0, workspace_text)

def clear_lab_pipeline_modules():
    for module_name in list(sys.modules):
        if module_name == 'lab_pipeline' or module_name.startswith('lab_pipeline.'):
            del sys.modules[module_name]

def find_uploaded_package(uploaded):
    for uploaded_name in uploaded:
        path = Path(uploaded_name)
        if path.suffix.lower() == '.zip' and path.stem.startswith(PACKAGE_ZIP_PREFIX):
            return path
    raise FileNotFoundError('Debes subir lab_pipeline_package.zip.')

def install_or_update_package(workspace_dir):
    package_folder = workspace_dir / 'lab_pipeline'
    if package_folder.exists():
        return
    if not running_in_colab():
        raise ModuleNotFoundError('No encontre lab_pipeline en la carpeta actual.')
    from google.colab import files
    print(f'Sube ahora {PACKAGE_ZIP}.')
    uploaded = files.upload()
    package_path = find_uploaded_package(uploaded)
    with zipfile.ZipFile(package_path, 'r') as z:
        z.extractall(workspace_dir)
    print(f'Paquete instalado en: {workspace_dir}')

mount_drive_if_colab()
WORKSPACE_DIR = find_notebook_workspace()
os.environ['LAB_PIPELINE_WORKSPACE_DIR'] = str(WORKSPACE_DIR)
install_or_update_package(WORKSPACE_DIR)
force_workspace_first(WORKSPACE_DIR)
clear_lab_pipeline_modules()
import lab_pipeline
print('lab_pipeline loaded.')
print(f'Workspace: {WORKSPACE_DIR}')
print(f'lab_pipeline file: {Path(lab_pipeline.__file__).resolve()}')


## 2. Launch Web App
Abre la aplicacion en un puerto. Si no se abre sola, usa el enlace que aparece en la salida.


In [ ]:
from lab_pipeline.web_app import launch_lab_manager_web_app

server = launch_lab_manager_web_app(port=7860)


## Actualizar paquete
Si te paso una version nueva del zip y ya existe `lab_pipeline`, borra la carpeta vieja o fuerza actualizacion con esta celda opcional.


In [ ]:
# Opcional: ejecutar solo cuando quieras actualizar lab_pipeline_package.zip
from google.colab import files
uploaded = files.upload()
package_path = find_uploaded_package(uploaded)
with zipfile.ZipFile(package_path, 'r') as z:
    z.extractall(WORKSPACE_DIR)
clear_lab_pipeline_modules()
print('Paquete actualizado. Vuelve a ejecutar Setup y Launch Web App.')
